In [ ]:
import pandas as pd 
import numpy as np  

import os

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
from keras.utils import to_categorical

In [ ]:
data = pd.read_hdf("../initialSingleCellDf-channel-20220916-MW_018-001.h5", key="df")
MARKERS = data.columns
# ANTIGENS = ['V4', 'T4', 'Q4', 'A2', 'N4']  # Only focus on the strong markers. 
ANTIGENS = ['null', 'E1', 'G4', 'V4', 'T4', 'Q4', 'A2', 'N4']
TIMES = [4.0, 12.0, 24.0, 30.0, 36.0, 48.0, 60.0, 72.0]
TEST_PROPORTION = 0.25

# %%
# Filter the data to the strong antigens, first repliate, OT-1 cells, high concentration, and later. 
df = data.loc[(data.index.get_level_values('CellType') == 'OT-1') & 
                       (data.index.get_level_values('Replicate') == '1') &
                       (data.index.get_level_values('Time') >= 30.0) & # [4.0, 12.0, 24.0, 30.0, 36.0, 48.0, 60.0, 72.0]
                       (data.index.get_level_values('Peptide').isin(ANTIGENS))
                    ]

group_size = 10

def avg(group): 
    group['GroupNumber'] = np.array(range(len(group.index))) // group_size
    res = group.groupby('GroupNumber').mean()
    return res

averaged_df = df.groupby(['Peptide', 'Time']).apply(avg)

print(averaged_df)
averaged_df = pd.DataFrame(StandardScaler().fit_transform(averaged_df), columns=averaged_df.columns, index=averaged_df.index) 

In [ ]:
selected_columns = ['FSC-A', 'SSC-A', 'CD25', 'CD38', 'Granzyme B', 'CD2', 'CD27', 'CD45RA',
       'CD4', 'CD86', 'CXCR6', 'CD5', 'CD62L', 'OX40', 'PD-L1', 'TBet',
       'CD126', 'Proliferation', 'ICOS', 'IRF8', 'CD19', 'MHC-II', 'CD45',
       'CD44', 'CX3CR1', 'CD8a']

# Extract the selected columns into a new DataFrame
selected_df = averaged_df[selected_columns].reset_index(drop=True)
print(selected_df.columns)
selected_df

In [ ]:
antigen = list(averaged_df.index.get_level_values('Peptide'))
time = list(averaged_df.index.get_level_values('Time'))

In [ ]:
X = np.array(averaged_df.values)
print(X.shape)
print(X)
y = np.array(list(map(lambda x: ANTIGENS.index(x), antigen)))
print("y: ", y)
y = to_categorical(y)
print("yshape", y.shape)
print(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_PROPORTION, random_state=42)

print(X_train.shape)
print(y_train.shape)

In [ ]:
# autoencoder = Sequential([
#     layers.InputLayer(input_shape=(26,)),
#     layers.Dense(13, activation='elu'),
#     layers.Dense(5, activation='relu'),
#     layers.Dense(2, activation='linear', name="Bottleneck"), # The bottleneck. 
#     layers.Dense(5, activation='relu'),
#     layers.Dense(13, activation='sigmoid'),
#     layers.Dense(26, activation='tanh'),
# ])
autoencoder = Sequential([
    layers.InputLayer(input_shape=(26,)),
    layers.Dense(13, activation='linear'),
    layers.Dense(7, activation='linear'),
    layers.Dense(3, activation='linear', name="Bottleneck"), # The bottleneck. 
    layers.Dense(7, activation='linear'),
    layers.Dense(13, activation='linear'),
    layers.Dense(26, activation='linear'),
])

In [ ]:
autoencoder.compile(optimizer='adagrad', loss=losses.MeanSquaredError(), metrics=['mse'])


In [ ]:
CHECKPOINT_PATH = "training/train_avg/cp-{epoch:04d}.ckpt"
checkpoint_dir = os.path.dirname(CHECKPOINT_PATH)

BATCH_SIZE = 1024
STEPS_PER_EPOCH = X_train.shape[0] / BATCH_SIZE
SAVE_PERIOD = 10


training_callback = ModelCheckpoint(filepath=CHECKPOINT_PATH,
                                    save_weights_only=True,
                                    verbose=1, 
                                    save_freq=int(SAVE_PERIOD * STEPS_PER_EPOCH)
                                )

# autoencoder.save_weights(CHECKPOINT_PATH.format(epoch=0))

tf.random.set_seed(42)

hist = autoencoder.fit(X_train, X_train,
                epochs=1000,
                shuffle=True,
                validation_data=(X_test, X_test),
                callbacks=[training_callback],
                batch_size=BATCH_SIZE,
                verbose=0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot the accuracy on the first subplot
ax1.plot(hist.history['mse'], label='Accuracy')
ax1.plot(hist.history['val_categorical_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

# Plot the loss on the second subplot
ax2.plot(hist.history['loss'], label='Loss')
ax2.plot(hist.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

# Adjust the spacing between subplots
plt.tight_layout()

In [ ]:
autoencoder.summary()

In [ ]:
train_time, test_time = train_test_split(time, test_size=TEST_PROPORTION, random_state=42)
train_antigen, test_antigen = train_test_split(antigen, test_size=TEST_PROPORTION, random_state=42)

# %%
import plotly.express as px

# Assuming you have X_test, embedding, test_antigen, test_time, and ANTIGENS defined

# Create a DataFrame with the embedding data
df = pd.DataFrame(embedding, columns=['Dimension 1', 'Dimension 2', 'Dimension 3'])
df['Antigen'] = ["null", "E1", "G4", "V4", "T4", "Q4", "A2", "N4"]

# Create an interactive 3D scatter plot
fig = px.scatter_3d(
    df,
    x='Dimension 1',
    y='Dimension 2',
    z='Dimension 3',
    color='Antigen',
    color_discrete_sequence=px.colors.qualitative.Set1,  # Define a color palette
    title="Autoencoder 3D Embedding Averaged Data (Validation Set)"
)

# Customize the appearance and interactivity if needed
fig.update_traces(marker=dict(size=4))
fig.update_layout(legend_title="Antigens")

# Show the plot
fig.show()

In [ ]:
MARKERS = ['SSC-A','CD25','Granzyme B','CD2','CD27','CD5','CD126','ICOS','IRF8','CD8a']

fig, axs = plt.subplots(2, 5, figsize=(15,5))

axs = axs.flatten()

vmin = -2 * np.std(averaged_df.values)
vmax = 2 * np.std(averaged_df.values)

for i, marker in enumerate(MARKERS):
    train_marker, test_marker = train_test_split(averaged_df[marker], test_size=TEST_PROPORTION, random_state=42)

    sc = axs[i].scatter(
        x=embedding[:,0], 
        y=embedding[:,1], 
        c=test_marker, 
        cmap='Blues', 
        s=1, 
        vmin=vmin,
        vmax=vmax
    )

    axs[i].set_title(marker)

plt.tight_layout()

# cax = fig.add_axes([0.92, 0.1, 0.02, 0.8])
plt.colorbar(sc, ax=axs.ravel().tolist())

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = np.argmax( autoencoder(X_test), axis=1 )
y_true = np.argmax(y_test, axis=1) 

cm = confusion_matrix(y_true, y_pred)

cm

In [ ]:
# Create a figure and axes
fig, ax = plt.subplots()

# Create the heatmap using seaborn
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)

# Set labels for the x and y axes
ax.set_xticklabels(ANTIGENS)
ax.set_yticklabels(ANTIGENS)